In [ ]:
import nltk
from nltk.tokenize import wordpunct_tokenize

In [ ]:
import unicodedata

def normalizar_texto(texto):

    texto = texto.lower().strip()

    texto = unicodedata.normalize("NFD", texto)

    texto = "".join(
        caracter for caracter in texto
        if unicodedata.category(caracter) != "Mn"
    )

    return texto

print(normalizar_texto("TITULACIÓN"))
print(normalizar_texto("Certificado de TERMINACIÓN"))
print(normalizar_texto("  Constancia  "))

In [ ]:
# Diccionario de intenciones
intenciones = {
    "saludo": ["hola", "buenas", "buenos"],
    "constancia": ["constancia", "constancias"],
    "certificado_parcial": ["certificado parcial", "certificados parciales"],
    "duplicado_certificado": ["duplicado de certificado", "duplicado del certificado"],
    "certificado_terminacion": ["certificado de terminacion","certificado de terminación"],
    "certificado": ["certificado", "certificados"],
    "titulacion": ["titulación", "titulacion", "título", "titulo", "titularme"],
    "despedida": ["adios", "adiós", "gracias", "bye"]
}

# Diccionario de respuestas
respuestas = {
    "saludo": "¡Hola! Soy el asistente de Servicios Escolares de Tecmilenio Campus Cancún. ¿En qué puedo ayudarte?",
    "constancia": "Puedo orientarte sobre el trámite de constancias de estudios.",
    "certificado_parcial": "Has seleccionado el trámite de certificado parcial. Puedo proporcionarte información sobre sus requisitos, costo y tiempo de emisión.",
    "duplicado_certificado": "Has seleccionado el trámite de duplicado de certificado total. Puedo proporcionarte información sobre sus requisitos, costo y tiempo de emisión.",
    "certificado_terminacion": "Has seleccionado el certificado de terminación de estudios. Puedo orientarte sobre este documento.",
    "certificado": "¿Qué tipo de certificado necesitas? 1. Certificado parcial. 2. Duplicado de certificado total. 3. Certificado de terminación.",
    "titulacion": "Puedo orientarte sobre el proceso de titulación.",
    "despedida": "¡Gracias por comunicarte con Servicios Escolares!"
}

intenciones.update({
    "recuperacion_documentos": [
        "recuperacion de documentos",
        "recuperacion de documentos de terminacion",
        "recuperar documentos"
    ],
    "historial_academico": [
        "historial academico",
        "historial de calificaciones"
    ],
    "transcript": [
        "transcript",
        "calificaciones en ingles"
    ],
    "constancia_autenticidad": [
        "constancia de autenticidad",
        "autenticidad"
    ],
    "apostilla": [
        "apostilla",
        "apostillar",
        "apostille"
    ],
    "transferencia": [
        "transferencia",
        "cambio de campus",
        "cambiar de campus",
        "cambiarme de campus",
        "transferirme de campus"
    ],
    "cambio_programa": [
        "cambio de programa",
        "cambiar de carrera"
    ],
    "baja_definitiva": [
        "baja definitiva",
        "baja"
    ],
    "reposicion_credencial": [
        "reposicion de credencial",
        "reponer mi credencial",
        "perdi mi credencial"
    ],
    "reposicion_marbete": [
        "reposicion de marbete",
        "reponer marbete",
        "perdi mi marbete"
    ]
})

respuestas.update({
    "recuperacion_documentos": "Aquí tienes la información para recuperar tus documentos de terminación electrónicos:",
    "historial_academico": "Aquí tienes la información para solicitar tu historial académico:",
    "transcript": "Aquí tienes la información para solicitar tu transcript:",
    "constancia_autenticidad": "Aquí tienes la información de la constancia de autenticidad:",
    "apostilla": "Aquí tienes la información del trámite de apostilla:",
    "transferencia": "Aquí tienes la información del trámite de transferencia:",
    "cambio_programa": "Aquí tienes la información para solicitar un cambio de programa:",
    "baja_definitiva": "Aquí tienes la información para solicitar una baja definitiva:",
    "reposicion_credencial": "Aquí tienes la información para reponer tu credencial:",
    "reposicion_marbete": "Aquí tienes la información para reponer tu marbete:"
})

In [ ]:
# Diccionario de imagenes
imagenes_tramites = {
    "constancia": "Diapositiva1.png",
    "certificado_parcial": "Diapositiva2.png",
    "duplicado_certificado": "Diapositiva3.png",
    "recuperacion_documentos": "Diapositiva4.png",
    "historial_academico": "Diapositiva5.png",
    "transcript": "Diapositiva6.png",
    "constancia_autenticidad": "Diapositiva7.png",
    "apostilla": "Diapositiva8.png",
    "transferencia": "Diapositiva9.png",
    "cambio_programa": "Diapositiva10.png",
    "baja_definitiva": "Diapositiva11.png",
    "reposicion_credencial": "Diapositiva12.png",
    "reposicion_marbete": "Diapositiva13.png"
}

In [ ]:
# Función para detectar intenciones
def detectar_intenciones(mensaje):

    mensaje = normalizar_texto(mensaje)
    palabras = wordpunct_tokenize(mensaje)
    texto = " ".join(palabras)

    intenciones_detectadas = []

    for intencion, expresiones in intenciones.items():
        for expresion in expresiones:

            expresion = normalizar_texto(expresion)

            if " " in expresion:
                encontrada = expresion in texto
            else:
                encontrada = expresion in palabras

            if encontrada:
                intenciones_detectadas.append(intencion)
                break

    if "certificado_parcial" in intenciones_detectadas:
        if "certificado" in intenciones_detectadas:
            intenciones_detectadas.remove("certificado")

    if "duplicado_certificado" in intenciones_detectadas:
        if "certificado" in intenciones_detectadas:
            intenciones_detectadas.remove("certificado")

    if "certificado_terminacion" in intenciones_detectadas:
        if "certificado" in intenciones_detectadas:
            intenciones_detectadas.remove("certificado")

    # Evitar que las constancias específicas activen la genérica
    if "constancia_autenticidad" in intenciones_detectadas:
        if "constancia" in intenciones_detectadas:
            intenciones_detectadas.remove("constancia")

    # Evitar mostrar el menú de certificados cuando
    # la consulta corresponde a otro trámite específico
    tramites_especificos = [
        "apostilla",
        "constancia_autenticidad",
        "recuperacion_documentos"
    ]

    if any(tramite in intenciones_detectadas for tramite in tramites_especificos):
        if "certificado" in intenciones_detectadas:
            intenciones_detectadas.remove("certificado")

    return intenciones_detectadas

In [ ]:
# Función para generar respuestas
from IPython.display import display, Image

def generar_respuestas(intenciones_detectadas):

    if not intenciones_detectadas:
        print("Chatbot: No comprendí tu consulta. Puedes preguntar sobre constancias, certificados o titulación.")

    else:
        for intencion in intenciones_detectadas:
            print("Chatbot:", respuestas[intencion])
            if intencion in imagenes_tramites:
                ruta_imagen = imagenes_tramites[intencion]
                display(Image(filename=ruta_imagen))

In [ ]:
# Función principal del chatbot
def iniciar_chatbot():

    print("=== CHATBOT DE SERVICIOS ESCOLARES ===")
    print("Escribe 'salir' para terminar la conversación.\n")

    tramite_pendiente = None

    while True:
        mensaje = input("Alumno: ").lower().strip()

        if mensaje == "salir":
          print("Chatbot: ¡Hasta pronto!")
          break

        if tramite_pendiente == "certificado":

            opciones = {
                "1": "certificado_parcial",
                "2": "duplicado_certificado",
                "3": "certificado_terminacion"
              }

            if mensaje in opciones:
                intencion = opciones[mensaje]
                generar_respuestas([intencion])
                tramite_pendiente = None
                continue

        resultado = detectar_intenciones(mensaje)

        if resultado == ["certificado"]:
          tramite_pendiente = "certificado"

        else:
          tramite_pendiente = None

        generar_respuestas(resultado)

# Iniciar el programa
iniciar_chatbot()

In [ ]:
pruebas = [
    ("Hola", "saludo"),
    ("Necesito una constancia", "constancia"),
    ("Quiero un certificado parcial", "certificado_parcial"),
    ("Necesito duplicado de certificado", "duplicado_certificado"),
    ("Quiero un transcript", "transcript"),
    ("Necesito mi historial académico", "historial_academico"),
    ("Quiero apostillar mi certificado", "apostilla"),
    ("Necesito una constancia de autenticidad", "constancia_autenticidad"),
    ("Quiero cambiar de campus", "transferencia"),
    ("Necesito reponer mi credencial", "reposicion_credencial")
]

aciertos = 0

for mensaje, esperado in pruebas:
    detectadas = detectar_intenciones(mensaje)
    correcto = esperado in detectadas

    if correcto:
        aciertos += 1

    print(f"Consulta: {mensaje}")
    print(f"Esperado: {esperado}")
    print(f"Detectado: {detectadas}")
    print(f"Resultado: {'CORRECTO' if correcto else 'INCORRECTO'}")
    print("-" * 50)

print(f"Resultado final: {aciertos}/{len(pruebas)} pruebas correctas")

In [ ]:
from datetime import datetime

nombre_archivo = "evidencias_chatbot.txt"

with open(nombre_archivo, "w", encoding="utf-8") as archivo:

    archivo.write("PRUEBAS FUNCIONALES DEL CHATBOT\n")
    archivo.write("Servicios Escolares - Tecmilenio Campus Cancún\n")
    archivo.write(f"Fecha: {datetime.now()}\n")
    archivo.write("=" * 55 + "\n\n")

    for numero, (mensaje, esperado) in enumerate(pruebas, start=1):

        detectadas = detectar_intenciones(mensaje)
        correcto = esperado in detectadas

        archivo.write(f"PRUEBA {numero}\n")
        archivo.write(f"Alumno: {mensaje}\n")

        if detectadas:
            for intencion in detectadas:
                archivo.write(
                    f"Chatbot: {respuestas[intencion]}\n"
                )

                if intencion in imagenes_tramites:
                    archivo.write(
                        f"Infografía: {imagenes_tramites[intencion]}\n"
                    )
        else:
            archivo.write(
                "Chatbot: No comprendí tu consulta.\n"
            )

        archivo.write(
            f"Resultado: {'CORRECTO' if correcto else 'INCORRECTO'}\n"
        )
        archivo.write("-" * 55 + "\n\n")

print("Archivo de evidencias generado correctamente.")

In [ ]:
from google.colab import files

files.download("evidencias_chatbot.txt")